# Atlas Cluster Visualization (fsaverage)

This pipeline renders **electrode clusters from multiple patients on a single standard atlas brain (fsaverage)**. It (1) merges cluster labels onto contact coordinates, (2) transforms each patient’s native coordinates into **fsaverage tkrRAS**, and (3) exports **high-resolution static PNGs** from standard viewpoints for a **clean** brain and an **aparc-parcellated** brain.



# Part A: Create Electrode Localization plots

## Part A.1: build atlas inputs into 240 from 230: transforms + caching

In Cell 1, I take the **sample-level clustering output** from the 230 run folder, specifically `df_keep_with_clusters.parquet` (located at `.../outputs/230_blob_clustering_runs/<RUN_ID_230>/df_keep_with_clusters.parquet`). From that table, I select the chosen cluster label column (e.g., `cluster_kmeans_blob_k36_q0.9`) and **collapse it to contact-level** by grouping on `(patient_id, electrode)` and taking a stable summary (mode) per contact. This produces a canonical contact-level metadata table `df_meta_contact_level.tsv`. Then, for each patient, I load that patient’s **native contact coordinates** (PAT/Micro from `Paper1_recons/<pid>/glassbrain/coords/<pid>_contacts_tkrRAS.csv`, EL from their Bern “Lookup.xlsx” source), and apply the **subject talairach.xfm + fsaverage talairach.xfm** to transform all contacts into **fsaverage tkrRAS**. The transformed coordinates are written to `electrode_coords_fsaverage_tkr.tsv`, and a per-patient success/failure report is written to `transform_qc_summary.tsv`. All three outputs are saved under the recon run folder: `.../outputs/240_blob_cluster_recon/<RUN_ID_RECON>/atlas_cache/`.

Files used (inputs):

* `.../outputs/230_blob_clustering_runs/<RUN_ID_230>/df_keep_with_clusters.parquet`
* Patient transforms: `<patient_fs_dir>/mri/transforms/talairach.xfm`
* Atlas reference: `fsaverage/mri/transforms/talairach.xfm`
* PAT/Micro native coords (when available): `Paper1_recons/<pid>/glassbrain/coords/<pid>_contacts_tkrRAS.csv`
* EL native coords: Bern `Lookup.xlsx` (sheet `channels`, native_x/y/z)

Files produced (outputs):

* `.../outputs/240_blob_cluster_recon/<RUN_ID_RECON>/atlas_cache/df_meta_contact_level.tsv`
* `.../outputs/240_blob_cluster_recon/<RUN_ID_RECON>/atlas_cache/electrode_coords_fsaverage_tkr.tsv`
* `.../outputs/240_blob_cluster_recon/<RUN_ID_RECON>/atlas_cache/transform_qc_summary.tsv`


In [1]:
import re, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import functions.lf_blob_recon as R


RUN_ID_230   = '20260111_191040' # "20260108_233820"            # the existing 230 run folder name
RUN_ID_RECON = RUN_ID_230#"kmeans_blob_20260102_135222"  # the 240 output folder name you want

# cluster column is algorithm-specific, but must start with "cluster_"
# If df_keep_with_clusters.parquet has exactly one such column, you can set CLUSTER_COL=None
CLUSTER_COL = 'cluster_kmeans_101_k23_q0p9'# 'cluster_kmeans_101_k25_q0p9' #'cluster_kmeans_101_k7_q0p85'# 'cluster_kmeans_blob_k16_q0p85' # 'cluster_kmeans_101_k7_q0p85'# None # "cluster_kmeans_valley_blob_bestK"  # or None if only one cluster_* col exists

def _safe_tag(s: str) -> str:
    """
    Make a filesystem-safe tag for folder naming.
    Keeps letters/numbers/_/- and replaces everything else with '_'.
    """
    s = str(s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "_", s)
    return s.strip("_")

# Use CLUSTER_COL name as the distinguishing recon folder tag
CLUSTER_TAG = _safe_tag(CLUSTER_COL) if CLUSTER_COL else "auto_cluster"

RUN_ID_RECON = f"{RUN_ID_230}__{CLUSTER_TAG}"

P = R.ensure_dirs(RUN_ID_230, RUN_ID_RECON)
print("230 meta in:", P["meta_in"])
print("240 out dir:", P["run240_dir"])
print("240 render out:", P["render_out"])

R.build_atlas_inputs(RUN_ID_230, RUN_ID_RECON, cluster_col=CLUSTER_COL)


230 meta in: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\230_blob_clustering_runs\20260111_191040\df_keep_with_clusters.parquet
240 out dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9
240 render out: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\renders_AB
[ERROR] EL030: Missing talairach.xfm: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\FreesurferResults\el030\mri\transforms\talairach.xfm
[ERROR] EL035: Missing talairach.xfm: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\FreesurferResults\el035\mri\transforms\talairach.xfm
[ERROR] EL036: Missing talairach.xfm: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\FreesurferResults\el036\mri\transforms\talairach.

{'base': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/scripts/functions'),
 'run_id_230': '20260111_191040',
 'run_id_recon': '20260111_191040__cluster_kmeans_101_k23_q0p9',
 'run230_dir': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/outputs/230_blob_clustering_runs/20260111_191040'),
 'meta_in': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/outputs/230_blob_clustering_runs/20260111_191040/df_keep_with_clusters.parquet'),
 'X_in': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/outputs/230_blob_clustering_runs/20260111_191040/X_blob_keep.npy'),
 'run240_dir': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/outputs/240_blob_cluster_recon/20260111_191040__cluster_kmeans_101_k23_q0p9'),
 'inputs_snapshot': WindowsPath('//nasac-m2.unige.ch/m-HumanNeur

## Part A.2: render A/B atlas figures into 240

In Cell 2, I load the two cached atlas inputs created in Cell 1: `df_meta_contact_level.tsv` (contact → cluster label) and `electrode_coords_fsaverage_tkr.tsv` (contact → fsaverage coordinates). I merge them on `(patient_id, electrode)`, optionally filter depth/subdural contacts depending on configuration, and then render **publication-style static PNGs** on the fsaverage cortical surface from standard viewpoints. I generate two figure styles: **(A) clean brain** (translucent cortex) and **(B) aparc brain** (Desikan–Killiany parcellation visible). In both cases, electrodes are plotted as spheres and **colored by cluster** using a stable colormap saved to disk. In addition, I save legend PNGs (cluster legend, plus global patient and condition legends) into the relevant output folders. Everything is written under `.../outputs/240_blob_cluster_recon/<RUN_ID_RECON>/renders_AB/` in a consistent subfolder structure.

Files used (inputs):

* `.../outputs/240_blob_cluster_recon/<RUN_ID_RECON>/atlas_cache/df_meta_contact_level.tsv`
* `.../outputs/240_blob_cluster_recon/<RUN_ID_RECON>/atlas_cache/electrode_coords_fsaverage_tkr.tsv`
* fsaverage surfaces: `fsaverage/surf/lh.pial`, `fsaverage/surf/rh.pial`
* aparc annotations: `fsaverage/label/lh.aparc.annot`, `fsaverage/label/rh.aparc.annot`

Files produced (outputs):

* Cluster/patient/condition color maps:

  * `.../renders_AB/cluster_color_map.json`
  * `.../renders_AB/patient_color_map.json`
  * `.../renders_AB/condition_color_map.json`
* Rendered PNGs, e.g.:

  * `.../renders_AB/clean_brain/all_clusters/clean_allclusters_left.png` (and other views)
  * `.../renders_AB/parcellated_aparc/all_clusters/aparc_allclusters_left.png` (and other views)
  * plus per-cluster subfolders if enabled
* Legend PNGs, e.g.:

  * `.../clean_brain/all_clusters/legend_clusters.png`
  * `.../clean_brain/all_clusters/legend_patients_global.png`
  * `.../clean_brain/all_clusters/legend_conditions_global.png`
  * and the same under `parcellated_aparc/all_clusters/`


In [65]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import functions.lf_blob_recon as R

# RUN_ID_230   = RUN_ID_230 # "20260103_122347"
# RUN_ID_RECON = RUN_ID_230

P = R.ensure_dirs(RUN_ID_230, RUN_ID_RECON)
print("240 render out:", P["render_out"])

R.render_atlas_figures(RUN_ID_230, RUN_ID_RECON)


240 render out: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\renders_AB

[QC] Merge summary
  - coords rows:            2208
  - missing cluster labels: 1955
  - missing any x/y/z:      0
  - kept 253/253 using 'isSubdural' with INCLUDE_DEPTH=True, INCLUDE_SUBDURAL=True
[INFO] Unique clusters: 18
[INFO] Unique patients: 13
[INFO] Unique conditions: 3 -> ['audio', 'picture', 'reading']
[WROTE] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\renders_AB\clean_brain\all_clusters\clean_allclusters_left.png
[WROTE] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\renders_AB\clean_brain\all_clusters\clean_allclusters_right.png
[WROTE] \\nasac-m2.unige.ch\m-HumanNeuro

{'base': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/scripts/functions'),
 'run_id_230': '20260111_191040',
 'run_id_recon': '20260111_191040__cluster_kmeans_101_k23_q0p9',
 'run230_dir': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/outputs/230_blob_clustering_runs/20260111_191040'),
 'meta_in': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/outputs/230_blob_clustering_runs/20260111_191040/df_keep_with_clusters.parquet'),
 'X_in': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/outputs/230_blob_clustering_runs/20260111_191040/X_blob_keep.npy'),
 'run240_dir': WindowsPath('//nasac-m2.unige.ch/m-HumanNeuronLab/ANALYSIS/FLM/Analysis_Lora/02_FBM_Clustering/outputs/240_blob_cluster_recon/20260111_191040__cluster_kmeans_101_k23_q0p9'),
 'inputs_snapshot': WindowsPath('//nasac-m2.unige.ch/m-HumanNeur

# Part B: Create PDFs: 
## PArt B.1: Get the ERSP centroids

In [66]:
P = R.ensure_dirs(RUN_ID_230, RUN_ID_RECON)
print(P["render_out"] / "clean_brain")

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\renders_AB\clean_brain


In [68]:
import functions.lf_blob_recon as R

# Save mean ERSP per cluster (clean, poster-ready PNGs)
out_dir = R.save_cluster_mean_ersp_pngs(
    RUN_ID_230,
    RUN_ID_RECON,
    cluster_col_in_keep=CLUSTER_COL,
    use_median=False,     # toggle mean vs median here
    fmax_hz=500.0,
    vmin=-6, vmax=6,
)
print("Saved ERSP cluster prototypes to:", out_dir)


Saved ERSP cluster prototypes to: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\reports\cluster_mean_ersp\cluster_kmeans_101_k23_q0p9


## Part B.2: Generate PDFs

In [69]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import functions.lf_blob_recon as R

# Which cluster column should the PDF use?
CLUSTER_COL_IN_KEEP = CLUSTER_COL

# ---- Preflight: confirm cluster IDs in meta and look for their render folders/files
P = R.ensure_dirs(RUN_ID_230, RUN_ID_RECON)

meta_keep_path = Path(P["meta_in"])
df_keep_with_clusters = pd.read_parquet(meta_keep_path)

if CLUSTER_COL_IN_KEEP is None:
    # Auto-detect if you want, but better to pass explicit column
    cand = [c for c in df_keep_with_clusters.columns if str(c).startswith("cluster_")]
    if len(cand) != 1:
        raise RuntimeError(f"CLUSTER_COL_IN_KEEP=None but found {len(cand)} cluster_* columns: {cand}")
    CLUSTER_COL_IN_KEEP = cand[0]

# cluster ids present
labs = df_keep_with_clusters[CLUSTER_COL_IN_KEEP].to_numpy()
labs = labs[np.isfinite(labs)].astype(int)
uniq = sorted(np.unique(labs).tolist())
print("[QC] Using cluster column:", CLUSTER_COL_IN_KEEP)
print("[QC] Unique cluster IDs:", uniq[:20], ("..." if len(uniq) > 20 else ""))
print("[QC] #clusters:", len(uniq))

render_root = Path(P["render_out"])

# Expected views (from recon config if available; else default set)
try:
    from functions import lf_blob_recon_config as C
    VIEWS = list(C.VIEWS_TO_SAVE)
except Exception:
    VIEWS = ["left", "right", "frontal", "posterior", "dorsal", "ventral"]

# Heuristic check: for each cluster id, do we find *any* png with that view name?
# This is robust even if the folder structure differs slightly.
missing = []
for cid in uniq:
    for v in VIEWS:
        # search for files containing both cluster id and view name
        hits = list(render_root.rglob(f"*{cid}*{v}*.png"))
        if len(hits) == 0:
            missing.append((cid, v))

if len(missing) == 0:
    print("[QC] All clusters have at least one PNG per view (heuristic).")
else:
    print(f"[QC] Missing (heuristic) {len(missing)} cluster-view combos. Example first 25:")
    print(missing[:25])
    print("[QC] This usually means:")
    print("  - You rendered under a different RUN_ID_RECON than you are using now, OR")
    print("  - Cluster IDs are offset (0-based vs 1-based) between meta and renders, OR")
    print("  - The render file naming does not include view strings as expected.")

# ---- Build PDF
pdf_path = R.build_cluster_diagnostics_pdf(
    RUN_ID_230,
    RUN_ID_RECON,
    cluster_col_in_keep=CLUSTER_COL_IN_KEEP,
)
print("PDF:", pdf_path)


[QC] Using cluster column: cluster_kmeans_101_k23_q0p9
[QC] Unique cluster IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19] ...
[QC] #clusters: 23
[QC] Missing (heuristic) 30 cluster-view combos. Example first 25:
[(13, 'left'), (13, 'right'), (13, 'frontal'), (13, 'posterior'), (13, 'dorsal'), (13, 'ventral'), (17, 'left'), (17, 'right'), (17, 'frontal'), (17, 'posterior'), (17, 'dorsal'), (17, 'ventral'), (20, 'left'), (20, 'right'), (20, 'frontal'), (20, 'posterior'), (20, 'dorsal'), (20, 'ventral'), (21, 'left'), (21, 'right'), (21, 'frontal'), (21, 'posterior'), (21, 'dorsal'), (21, 'ventral'), (22, 'left')]
[QC] This usually means:
  - You rendered under a different RUN_ID_RECON than you are using now, OR
  - Cluster IDs are offset (0-based vs 1-based) between meta and renders, OR
  - The render file naming does not include view strings as expected.
[WROTE] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blo

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\scripts\functions\lf_blob_recon.py:1547: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\scripts\functions\lf_blob_recon.py:1647: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\scripts\functions\lf_blob_recon.py:1647: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\scripts\functions\lf_blob_recon.py:1647: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be inco

[WROTE] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\reports\cluster_diagnostics_report.pdf
[WROTE] ERSP prototypes: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\reports\cluster_ersp_prototypes\cluster_kmeans_101_k23_q0p9
PDF: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\reports\cluster_diagnostics_report.pdf


## PART B.3: Make full PDF render: 

In [1]:
# import sys
# from pathlib import Path
# sys.path.insert(0, str(Path.cwd()))

# import functions.lf_blob_recon as R

# CLUSTER_COL_IN_KEEP = CLUSTER_COL  # keep consistent

# pdf_path = R.build_cluster_diagnostics_pdf(
#     RUN_ID_230,
#     RUN_ID_RECON,
#     cluster_col_in_keep=CLUSTER_COL_IN_KEEP,
# )
# print("PDF:", pdf_path)

In [ ]:
# # make_cluster_report_pdf.py
# # PDF report generator for Cell 4 (ReportLab-based)
# #
# # Updated to respect the NEW folder organization:
# #   \\...\\02_FBM_Clustering\\outputs\\240_blob_cluster_recon\\<RUN_ID_RECON>\\
# #       atlas_cache\\...
# #       renders_AB\\...
# #       reports\\...  (PDF goes here)
# #
# # It scans renders_AB to discover clusters and then builds:
# #   1) Optional overview pages (heatmaps PNG if present)
# #   2) Optional montage (clean brain, left view, per cluster)
# #   3) Per-cluster pages:
# #        - colored by patient (clean+aparc, left/right)
# #        - colored by condition (clean+aparc, left/right)
# #
# # This script is robust to slight variations in your per-cluster folder/file naming by
# # searching for PNGs that contain: cluster number + view name (+ bypatient/bycondition if possible).



# from __future__ import annotations

# import re
# from pathlib import Path
# from typing import List, Optional, Tuple, Dict

# from reportlab.lib.pagesizes import A4, landscape
# from reportlab.lib.units import mm
# from reportlab.pdfgen import canvas
# from reportlab.lib.utils import ImageReader
# from PIL import Image


# # -----------------------------
# # Config (edit these)
# # -----------------------------
# RUN240_DIR = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon",RUN_ID_RECON)

# RENDER_DIRNAME = "renders_AB"
# REPORTS_DIRNAME = "reports"

# # Views to place on pages (2 views per 4-up page)
# VIEWS_2UP: Tuple[str, str] = ("left", "right")

# # Optional: include a heatmaps overview page if a PNG exists in reports/
# # (e.g., created by your Cell 3 heatmap step)
# HEATMAP_PNG_CANDIDATES = [
#     "cluster_diagnostics_heatmaps.png",
#     "cluster_heatmaps.png",
#     "heatmaps_overview.png",
# ]

# # Optional: montage settings
# MAKE_MONTAGE_PAGE = True
# MONTAGE_NCOLS = 5
# MONTAGE_THUMB_VIEW = "left"   # which view to use for montage thumbnails
# MONTAGE_STYLE = "clean"       # "clean" or "aparc"


# # -----------------------------
# # Helper: image sizing
# # -----------------------------
# def _img_size(path: Path) -> Tuple[int, int]:
#     with Image.open(path) as im:
#         return im.size  # (w, h)


# def draw_image_fit(c: canvas.Canvas, img_path: Path, x: float, y: float, w: float, h: float):
#     """Draw image preserving aspect ratio to fit within w x h box."""
#     iw, ih = _img_size(img_path)
#     if iw == 0 or ih == 0:
#         return
#     ar = iw / ih
#     box_ar = w / h
#     if ar >= box_ar:
#         ww = w
#         hh = w / ar
#     else:
#         hh = h
#         ww = h * ar
#     xx = x + (w - ww) / 2
#     yy = y + (h - hh) / 2
#     c.drawImage(ImageReader(str(img_path)), xx, yy, ww, hh, mask="auto")


# # -----------------------------
# # Discover clusters + locate images
# # -----------------------------
# _CLUSTER_RE = re.compile(r"(?:^|[_\-])cluster[_\-]?(\d+)(?:$|[_\-])", re.IGNORECASE)


# def discover_clusters(render_root: Path) -> List[int]:
#     """
#     Robustly finds clusters by scanning for folder names or file names containing 'cluster_<int>'.
#     Works with either of these typical layouts:
#       renders_AB/clean_brain/one_cluster/cluster_12/...
#       renders_AB/clean_brain/cluster_12/...
#       renders_AB/.../clean_cluster12_left.png
#     """
#     clusters = set()

#     # 1) folder-based discovery
#     for p in render_root.rglob("*"):
#         if p.is_dir():
#             m = _CLUSTER_RE.search(p.name)
#             if m:
#                 clusters.add(int(m.group(1)))

#     # 2) file-based discovery (fallback)
#     for p in render_root.rglob("*.png"):
#         m = _CLUSTER_RE.search(p.stem)
#         if m:
#             clusters.add(int(m.group(1)))

#     return sorted(clusters)


# def _style_dir(render_root: Path, style: str) -> Path:
#     """
#     Map style tag to directory name.
#     Your pipeline uses:
#       clean_brain/
#       parcellated_aparc/
#     """
#     style = style.lower().strip()
#     if style in ("clean", "clean_brain"):
#         return render_root / "clean_brain"
#     if style in ("aparc", "parcellated", "parcellated_aparc"):
#         return render_root / "parcellated_aparc"
#     raise ValueError(f"Unknown style: {style}")


# def _cluster_dir_candidates(style_root: Path, cluster: int) -> List[Path]:
#     """
#     Generate likely cluster folder paths.
#     We try with and without zero-padding and with 'one_cluster' subfolder (common in your code).
#     """
#     c2 = f"{cluster:02d}"
#     candidates = [
#         style_root / "one_cluster" / f"cluster_{c2}",
#         style_root / "one_cluster" / f"cluster_{cluster}",
#         style_root / f"cluster_{c2}",
#         style_root / f"cluster_{cluster}",
#     ]
#     # Also allow the case where cluster folders are nested deeper
#     # (we will still rglob later if needed).
#     return candidates


# def _find_png_in_folder(folder: Path, cluster: int, view: str, mode_hint: Optional[str]) -> Optional[Path]:
#     """
#     Search for a PNG inside folder that matches:
#       - contains the view token (e.g., 'left')
#       - contains the cluster number somewhere
#       - if mode_hint provided: prefer files containing 'bypatient'/'bycondition'/'bycluster'
#     """
#     if not folder.exists():
#         return None

#     view = view.lower()
#     c2 = f"{cluster:02d}"
#     c1 = str(cluster)

#     pngs = list(folder.glob("*.png"))
#     if not pngs:
#         return None

#     def score(p: Path) -> int:
#         s = p.stem.lower()
#         sc = 0
#         if view in s:
#             sc += 10
#         if f"cluster{c2}" in s or f"cluster_{c2}" in s:
#             sc += 8
#         if f"cluster{c1}" in s or f"cluster_{c1}" in s:
#             sc += 6
#         if mode_hint and mode_hint.lower() in s:
#             sc += 5
#         # prefer clean/aparc tags if present
#         if "clean" in s:
#             sc += 1
#         if "aparc" in s:
#             sc += 1
#         return sc

#     pngs_sorted = sorted(pngs, key=score, reverse=True)
#     best = pngs_sorted[0]
#     # Require at least view match; if not, reject and let caller fallback to broader search.
#     if view not in best.stem.lower():
#         return None
#     return best


# def pick_view_png(render_root: Path, style: str, cluster: int, mode: str, view: str) -> Optional[Path]:
#     """
#     Find the best PNG for a (style, cluster, mode, view).
#     mode options (expected by your pipeline): 'by_patient', 'by_condition', 'by_cluster'
#     We accept mode aliases: bypatient/bycondition/bycluster.
#     """
#     style_root = _style_dir(render_root, style)
#     mode_norm = mode.lower().strip()
#     mode_hint = None
#     if mode_norm in ("by_patient", "bypatient"):
#         mode_hint = "bypatient"
#         mode_dirs = ["by_patient", "byPatient", "by_patient_id", "bypatient", "patient", "patients"]
#     elif mode_norm in ("by_condition", "bycondition"):
#         mode_hint = "bycondition"
#         mode_dirs = ["by_condition", "byCondition", "bycondition", "condition", "conditions"]
#     elif mode_norm in ("by_cluster", "bycluster"):
#         mode_hint = "bycluster"
#         mode_dirs = ["by_cluster", "byCluster", "bycluster", "cluster"]
#     else:
#         mode_hint = None
#         mode_dirs = [mode]

#     # First: try likely cluster folder layouts + explicit mode subfolders
#     for cdir in _cluster_dir_candidates(style_root, cluster):
#         for md in mode_dirs:
#             p = _find_png_in_folder(cdir / md, cluster, view, mode_hint=mode_hint)
#             if p:
#                 return p
#         # Also try directly inside cluster folder (if no mode subfolder)
#         p = _find_png_in_folder(cdir, cluster, view, mode_hint=mode_hint)
#         if p:
#             return p

#     # Fallback: global search within style_root for any file that includes cluster + view (+ mode)
#     view_l = view.lower()
#     patt = re.compile(rf".*cluster[_\-]?0*{cluster}\b.*{re.escape(view_l)}.*\.png$", re.IGNORECASE)
#     candidates = []
#     for p in style_root.rglob("*.png"):
#         s = p.name.lower()
#         if view_l not in s:
#             continue
#         if f"cluster_{cluster}" not in s and f"cluster{cluster}" not in s and f"cluster_{cluster:02d}" not in s and f"cluster{cluster:02d}" not in s:
#             continue
#         if mode_hint and mode_hint not in s:
#             # do not hard-reject; we only prefer
#             pass
#         candidates.append(p)

#     if not candidates:
#         return None

#     # Prefer those with mode_hint if available
#     if mode_hint:
#         with_hint = [p for p in candidates if mode_hint in p.name.lower()]
#         if with_hint:
#             candidates = with_hint

#     # Prefer those closer to "one_cluster/cluster_xx"
#     def depth_score(p: Path) -> int:
#         parts = [x.lower() for x in p.parts]
#         sc = 0
#         if "one_cluster" in parts:
#             sc += 3
#         if any(x.startswith("cluster_") for x in parts):
#             sc += 2
#         return sc

#     candidates = sorted(candidates, key=depth_score, reverse=True)
#     return candidates[0]


# # -----------------------------
# # PDF layout
# # -----------------------------
# def add_title(c: canvas.Canvas, title: str, subtitle: str = ""):
#     c.setFont("Helvetica-Bold", 16)
#     c.drawString(18 * mm, 195 * mm, title)
#     if subtitle:
#         c.setFont("Helvetica", 10)
#         c.drawString(18 * mm, 190 * mm, subtitle)


# def add_page_4up(c: canvas.Canvas, title: str, items: List[Tuple[str, Optional[Path]]]):
#     """
#     4-up layout (2x2): each item is (label, image_path)
#     """
#     add_title(c, title)
#     boxes = [
#         (15 * mm, 105 * mm, 135 * mm, 80 * mm),   # top-left
#         (155 * mm, 105 * mm, 135 * mm, 80 * mm),  # top-right
#         (15 * mm, 15 * mm, 135 * mm, 80 * mm),    # bottom-left
#         (155 * mm, 15 * mm, 135 * mm, 80 * mm),   # bottom-right
#     ]
#     c.setFont("Helvetica", 9)
#     for (label, img), (x, y, w, h) in zip(items, boxes):
#         c.drawString(x, y + h + 2 * mm, label)
#         if img and img.exists():
#             draw_image_fit(c, img, x, y, w, h)
#         else:
#             c.setFont("Helvetica-Oblique", 9)
#             c.drawString(x, y + h / 2, "MISSING")
#             c.setFont("Helvetica", 9)


# def add_overview_montage_page(
#     c: canvas.Canvas,
#     title: str,
#     cluster_thumbs: List[Tuple[int, Path]],
#     ncols: int = 5,
# ):
#     """
#     Grid of thumbnails for quick scan.
#     """
#     add_title(c, title, f"Overview montage ({MONTAGE_STYLE}, {MONTAGE_THUMB_VIEW} view)")
#     x0, y0 = 15 * mm, 15 * mm
#     page_w, page_h = landscape(A4)
#     usable_w = page_w - 2 * x0
#     usable_h = 170 * mm  # leave space for title
#     cell_w = usable_w / ncols
#     cell_h = cell_w * 0.75

#     c.setFont("Helvetica", 8)
#     for i, (cl, img) in enumerate(cluster_thumbs):
#         row = i // ncols
#         col = i % ncols
#         x = x0 + col * cell_w
#         y = y0 + usable_h - (row + 1) * cell_h
#         c.drawString(x, y + cell_h - 4 * mm, f"Cluster {cl:02d}")
#         draw_image_fit(c, img, x, y, cell_w, cell_h - 6 * mm)


# def add_single_image_page(c: canvas.Canvas, title: str, img_path: Path, subtitle: str = ""):
#     add_title(c, title, subtitle)
#     x, y, w, h = 15 * mm, 15 * mm, 275 * mm, 170 * mm
#     if img_path.exists():
#         draw_image_fit(c, img_path, x, y, w, h)
#     else:
#         c.setFont("Helvetica-Oblique", 11)
#         c.drawString(x, y + h / 2, f"MISSING: {img_path.name}")


# # -----------------------------
# # Main report generator (Cell 4)
# # -----------------------------
# def build_cluster_pdf(
#     run240_dir: Path,
#     out_pdf: Optional[Path] = None,
#     views: Tuple[str, str] = ("left", "right"),
# ) -> Path:
#     """
#     Produces a PDF containing:
#       - Optional heatmaps overview page (if PNG exists in reports/)
#       - Optional montage (clean/aparc, left view per cluster)
#       - Per cluster:
#           - by_patient page (clean + aparc, left/right)
#           - by_condition page (clean + aparc, left/right)

#     Folder expectations (new organization):
#       run240_dir/
#         renders_AB/
#           clean_brain/...
#           parcellated_aparc/...
#         reports/
#           <PDF written here>
#     """
#     run240_dir = Path(run240_dir)
#     render_root = run240_dir / RENDER_DIRNAME
#     reports_dir = run240_dir / REPORTS_DIRNAME

#     if not render_root.exists():
#         raise FileNotFoundError(f"Missing render root: {render_root}")

#     clusters = discover_clusters(render_root)
#     if not clusters:
#         raise RuntimeError(f"No clusters found under: {render_root}")

#     reports_dir.mkdir(parents=True, exist_ok=True)

#     if out_pdf is None:
#         out_pdf = reports_dir / f"cluster_report_{run240_dir.name}.pdf"
#     else:
#         out_pdf = Path(out_pdf)
#         out_pdf.parent.mkdir(parents=True, exist_ok=True)

#     c = canvas.Canvas(str(out_pdf), pagesize=landscape(A4))

#     # --- Optional heatmaps page (if present) ---
#     heatmap_png = None
#     for name in HEATMAP_PNG_CANDIDATES:
#         p = reports_dir / name
#         if p.exists():
#             heatmap_png = p
#             break
#     if heatmap_png is not None:
#         add_single_image_page(
#             c,
#             title=f"Cluster diagnostics — {run240_dir.name}",
#             subtitle="Heatmaps overview",
#             img_path=heatmap_png,
#         )
#         c.showPage()

#     # --- Optional montage page ---
#     if MAKE_MONTAGE_PAGE:
#         thumbs: List[Tuple[int, Path]] = []
#         for cl in clusters:
#             p = pick_view_png(render_root, MONTAGE_STYLE, cl, mode="by_cluster", view=MONTAGE_THUMB_VIEW)
#             if p is None:
#                 # fallback: try by_patient
#                 p = pick_view_png(render_root, MONTAGE_STYLE, cl, mode="by_patient", view=MONTAGE_THUMB_VIEW)
#             if p is not None:
#                 thumbs.append((cl, p))

#         if thumbs:
#             add_overview_montage_page(c, f"Atlas cluster report — {run240_dir.name}", thumbs, ncols=MONTAGE_NCOLS)
#             c.showPage()

#     # --- Per-cluster pages ---
#     for cl in clusters:
#         # by_patient
#         items = [
#             (f"Clean • {views[0]}", pick_view_png(render_root, "clean", cl, "by_patient", views[0])),
#             (f"Clean • {views[1]}", pick_view_png(render_root, "clean", cl, "by_patient", views[1])),
#             (f"Aparc • {views[0]}", pick_view_png(render_root, "aparc", cl, "by_patient", views[0])),
#             (f"Aparc • {views[1]}", pick_view_png(render_root, "aparc", cl, "by_patient", views[1])),
#         ]
#         add_page_4up(c, f"Cluster {cl:02d} — colored by patient", items)
#         c.showPage()

#         # by_condition
#         items = [
#             (f"Clean • {views[0]}", pick_view_png(render_root, "clean", cl, "by_condition", views[0])),
#             (f"Clean • {views[1]}", pick_view_png(render_root, "clean", cl, "by_condition", views[1])),
#             (f"Aparc • {views[0]}", pick_view_png(render_root, "aparc", cl, "by_condition", views[0])),
#             (f"Aparc • {views[1]}", pick_view_png(render_root, "aparc", cl, "by_condition", views[1])),
#         ]
#         add_page_4up(c, f"Cluster {cl:02d} — colored by condition", items)
#         c.showPage()

#     c.save()
#     print(f"[WROTE] {out_pdf}")
#     return out_pdf


# if __name__ == "__main__":
#     # By default, use the RUN240_DIR defined at the top.
#     # You can also pass a different run240_dir by editing RUN240_DIR.
#     build_cluster_pdf(RUN240_DIR, out_pdf=None, views=VIEWS_2UP)


# Part C: Dynamic power (dB) brain maps across time

What I assumed (so you can correct quickly if needed)

- Your 230 run folder contains ersp_keep.npy or ersp_keep.npz aligned with df_keep_with_clusters.parquet.

- Your 240 atlas cache already contains electrode_coords_fsaverage_tkr.tsv (from build_atlas_inputs()).

- “Average value of activity” means kernel-weighted average across nearby contacts, after aggregating multiple samples per contact (mean/median).

### Outline

#### `render_surface_activity_maps_for_bin(RUN_ID_230, RUN_ID_RECON, f_bin, t_bin, ...)` does the following

* **Loads inputs**

  * `df_keep_with_clusters.parquet` from the **230 run** (`P["meta_in"]`)
  * `ersp_keep.npy` (preferred) or `ersp_keep.npz` from the **230 run**
  * `electrode_coords_fsaverage_tkr.tsv` from the **240 atlas cache** (`P["coords_out"]`)
  * fsaverage meshes (`lh.pial`, `rh.pial`) via `load_fsaverage_meshes()`
* **Extracts one ERSP scalar per sample**

  * takes the value at `[f_bin, t_bin]` for each sample
  * optionally filters samples by `condition="audio"/...`
* **Aggregates sample-level values to contact-level**

  * groups by `(patient_id, electrode)`
  * computes **mean** or **median** value per contact (your choice)
* **Projects contact values onto the cortical surface**

  * finds nearby surface vertices within `radius_mm`
  * applies a Gaussian kernel (`sigma_mm`) to compute a **weighted average activity per vertex**
  * optionally **drops contacts far from pial** (`exclude_contacts_dist_to_pial_mm_gt`, e.g., 12 mm)
* **Converts per-vertex activity into colors**

  * maps activity to `bwr` with `vmin=-6`, `vmax=+6` (clipped)
* **Optionally blends in a density/coverage overlay**

  * computes a per-vertex “coverage weight” (how much kernel mass hits each vertex)
  * converts that to a **black→white** grayscale image
  * blends it into the activity colors using `density_mix` and `density_gamma`
* **Renders and saves PNGs**

  * saves `<tag>_<view>.png` for each view in `C.VIEWS_TO_SAVE`
  * output folder:
    `240/<RUN_ID_RECON>/reports/surface_activity_maps/`
* **Caches the numeric fields**

  * saves `240/<RUN_ID_RECON>/atlas_cache/surface_activity_cache/<tag>_fields.npz`
  * contains `act_lh, act_rh, den_lh, den_rh` and all parameters used

---

#### `_compute_contact_values_for_bin(df_keep, ersp, f_bin, t_bin, condition, agg_within_contact)` does the following

* **Checks alignment**: verifies `len(df_keep) == ersp.shape[0]`
* **Pulls the scalar** `ersp[:, f_bin, t_bin]` per sample
* **Optionally filters** by `condition`
* **Aggregates** sample values to one value per `(patient_id, electrode)` using mean/median
* **Returns** a compact table: `patient_id, electrode, value`

---

#### `_project_contacts_to_surface(df_vals, df_coords_fsavg, lh_pts, rh_pts, radius_mm, sigma_mm, ...)` does the following

* **Merges** `df_vals` with fsaverage coords on `(patient_id, electrode)`
* **Computes distance-to-pial** (nearest vertex distance to LH and RH)
* **Optionally filters** contacts with `dist_to_pial_mm` above a threshold
* **Assigns each contact to a hemisphere** (whichever surface is closer)
* **Spreads each contact’s value** onto nearby vertices using a Gaussian kernel
* **Outputs**

  * `act_lh/act_rh`: weighted mean activity per vertex
  * `den_lh/den_rh`: “density” (sum of weights) per vertex

---

#### `_rgba_from_activity(values, vmin, vmax, cmap_name="bwr")` does the following

* maps numeric activity values to RGBA using a colormap and clipping

---

#### `_rgba_from_density_gray(density, gamma)` and `_blend_rgba(activity_rgba, density_rgba, mix)` do the following

* converts density to a black→white grayscale RGBA (with gamma shaping for visibility)
* blends the grayscale into activity colors (so coverage can be visible without destroying sign)

---

##### What your attached brain image represents (in plain terms)

* The **blue patches** are vertices where the projected activity is **negative** (toward `-6`).
* The **red patches** are vertices where the projected activity is **positive** (toward `+6`).
* The **gray/neutral areas** are near zero or have minimal contribution.
* If `density_gray=True` with blending:

  * regions with **low coverage** are pushed toward **black-ish/low-intensity**
  * regions with **high coverage** are pushed toward **white-ish/high-intensity**
  * the goal is: you can visually tell where you *actually had sampling support*.


In [36]:
import functions.lf_blob_recon as R
import re

RUN_ID_230   = "20260111_191040"


CLUSTER_COL = 'cluster_kmeans_101_k23_q0p9'# 'cluster_kmeans_101_k25_q0p9' #'cluster_kmeans_101_k7_q0p85'# 'cluster_kmeans_blob_k16_q0p85' # 'cluster_kmeans_101_k7_q0p85'# None # "cluster_kmeans_valley_blob_bestK"  # or None if only one cluster_* col exists

def _safe_tag(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))
    return s.strip("_")

CLUSTER_TAG = _safe_tag(CLUSTER_COL) if CLUSTER_COL else "auto_cluster"

RUN_ID_RECON = f"{RUN_ID_230}__{CLUSTER_TAG}"
f_bands = [[0,5],[5,10],[10,16],[16.40],[40,70],[70,130],[130,250],[250,500]]
f_bands = [[f_band1*500/129,f_band2*500/129] for f_band1,f_band2 in f_bands]
# f_bands = [f_band[0] for f_band in f_bands]
print(f_bands)
j_inc = 30
i_inc = 10
for j in range (0,300,j_inc):
    for f_band in f_bands:
        print(i,j)
        # Example: single bin
        R.render_surface_activity_from_ersp_window(
            RUN_ID_230, RUN_ID_RECON,
            f_bins=f_band, t_bins=(j,min(j+j_inc-1,300)),
            # condition="picture", #["picture","audio","reading"],           # or None for all
            vmin=-6, vmax=6,
            k_nearest = 4,                        # contacts used per vertex (KDTree query)
            sigma_mm=4.0,
            density_saturation=True,
            density_gamma=0.6,
            s_min = 0.15,
            s_max = 1.25,
            
        )


ValueError: not enough values to unpack (expected 2, got 1)

Below is what each parameter changes in the **output** of `render_surface_activity_from_ersp_window(...)`, in practical terms. I’m describing the behavior as it is typically implemented in your current pipeline: **ERSP window → per-contact summary → interpolate to surface vertices (kNN) → optional surface smoothing → color-map with vmin/vmax → optional density modulation → save PNGs per view**.

---

## Run + IO

**`run_id_230`**

* Chooses the *input* run folder (loads `df_keep_with_clusters.parquet` and `ersp_keep.npy/.npz` from the 230 run).
* If this points to the wrong run, you’ll either get missing-file errors or mismatched sample counts.

**`run_id_recon`**

* Chooses the *output* recon folder under `240_blob_cluster_recon/<run_id_recon>/...`.
* Controls where images and caches are written (so you can keep multiple recon variants separated).

**`out_subdir`**

* Names the subfolder under `.../reports/` (or wherever you write figures).
* Useful to avoid overwriting when you sweep many windows.

**`views`**

* Which camera viewpoints get saved (`left/right/frontal/...`).
* Reduces runtime and file clutter if you limit to a small set.

---

## What data is being summarized

**`condition`**

* Filters which samples contribute to the map (e.g., only “picture”).
* If `None`: uses all samples.
* If the condition column naming/values don’t match your meta, you may unintentionally select zero samples (leading to blank/near-zero maps).

**`f_bins`**

* Selects the **frequency range** used to compute the per-sample ERSP summary that drives the surface coloring.
* If you pass a *range*, the function averages (or otherwise aggregates) over that band.
* Important: your implementation must be consistent about whether these are **bin indices** or **Hz**. If you pass floats like `[70,130]` and the function expects indices, you’ll get wrong regions or out-of-range behavior.

**`t_bins`**

* Selects the **time-bin range** used to compute the per-sample ERSP summary.
* Same comment as `f_bins`: must match whether the function expects *bin indices* or *time units*.

**`agg_contact`** (if present in your signature)

* Defines how multiple samples belonging to the **same contact** are collapsed to one value before projection to the surface.
* Typical options:

  * `"mean"`: smooth, but can cancel positive/negative.
  * `"median"`: robust to outliers; less cancellation than mean in some cases.
  * `"absmean"` / separate pos/neg (if you implemented): reduces cancellation but changes interpretation.

---

## How values are mapped from contacts to the surface

**`k_nearest`**

* Controls how many contacts contribute to each surface vertex (kNN interpolation).
* Larger `k_nearest`:

  * smoother, more global looking maps
  * but more mixing/cancellation (more “white” around 0 if using `bwr`)
* Smaller `k_nearest`:

  * more focal, patchy maps
  * more sensitive to sparse sampling / noise

**`sigma_mm`**

* Surface smoothing scale (Gaussian kernel or equivalent).
* Higher `sigma_mm`:

  * blurrier, larger contiguous blobs
  * more cancellation toward 0 (more white with `bwr`)
* Lower `sigma_mm`:

  * sharper, more local structure
  * can look speckled if sampling is sparse

**`exclude_contacts_dist_to_pial_mm_gt`** (if you’re using it)

* Excludes contacts whose **distance to pial** is greater than this threshold.
* Lower threshold:

  * keeps only near-cortical contacts (more “surface-relevant”)
  * but may remove too much data (more low-density areas)
* Higher threshold:

  * includes deeper contacts (more coverage)
  * but mapping to pial surface becomes less anatomically meaningful for depth-only effects

---

## Colormap and intensity scaling

**`vmin`, `vmax`**

* Hard limits for color scaling.
* Narrower range (e.g., -3..3):

  * higher apparent contrast, easier to see subtle effects
  * but saturates extremes quickly
* Wider range (e.g., -10..10):

  * less saturation
  * but most values look closer to 0 → more white-ish with `bwr`

**`cmap_name`**

* Which colormap is used (`"bwr"` makes **0 map to white**).
* With `"bwr"`, white is *expected* whenever the projected value is near 0.

---

## Density modulation (your “black = low density, light = high density” layer)

**`density_saturation`**

* Toggle for applying density-based modulation.
* If `False`: color reflects only the activity value.
* If `True`: the activity color is modulated by local sampling density (how well a vertex is supported by nearby contacts).

**`density_gamma`**

* Adjusts the *contrast curve* of density.
* `gamma < 1` (e.g., 0.6):

  * boosts differences at the low-density end (more separation between sparse vs medium)
* `gamma > 1`:

  * compresses low-density differences; emphasizes only very high density

**`density_norm_q`** (if present)

* Sets what “high density” means by using a quantile for normalization.
* Lower quantile:

  * easier to reach “high density” (more of the brain looks high-density)
* Higher quantile:

  * only the best-covered areas reach high density (stronger blackening of sparse regions)

**`s_min`, `s_max`**

* Clamps the density scaling factor.
* Higher `s_min`:

  * prevents low-density areas from going too dark/attenuated
* Lower `s_min`:

  * makes sparse areas much darker / less visible
* Lower `s_max`:

  * prevents high-density areas from becoming too “bright/washed”
* Higher `s_max`:

  * allows high-density areas to look much more intense (sometimes “too white” depending on implementation)

---

## What this function does *not* do (important)

* It does **not** validate anatomical correctness of mapping by itself (you need QC checks like nearest-vertex distance distributions, hemisphere sanity checks, etc.).
* It does **not** guarantee “no white” if you use `bwr` with symmetric limits; white is the zero-point color.
* It does **not** solve depth-electrode volumetric labeling; it’s a surface projection/visualization.

---

If you paste the **exact function signature** from your current `lf_blob_recon.py` (just the `def render_surface_activity_from_ersp_window(...):` line), I can rewrite this as a parameter-by-parameter docstring that matches your implementation exactly (including whether `f_bins/t_bins` are interpreted as indices or Hz/%).


In [7]:
import re
from pathlib import Path

import numpy as np
import functions.lf_blob_recon as R

# -------------------------
# CONFIG
# -------------------------
RUN_ID_230 = "20260111_191040"

CLUSTER_COL = "cluster_kmeans_101_k23_q0p9"  # used only to tag RUN_ID_RECON

FMAX_HZ = 500.0  # must match your ERSP frequency axis convention

# time window step (in time bins)
T_INC = 30

# optional: loop conditions; set to [None] for all samples pooled
CONDITIONS = ["picture", "audio", "reading"]  # e.g. [None, "picture", "audio", "reading"]

# frequency bands in Hz (low, high)
F_BANDS_HZ = [
    (0, 5),
    (5, 10),
    (10, 16),
    (16, 40),
    (40, 70),
    (70, 130),
    (130, 250),
    (250, 500),
]

# -------------------------
# Helpers
# -------------------------
def _safe_tag(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))
    return s.strip("_")

def hz_to_bin(hz: float, nF: int, fmax_hz: float) -> int:
    """Map Hz -> nearest frequency bin index in [0, nF-1], assuming linear 0..fmax_hz."""
    hz = float(hz)
    hz = max(0.0, min(hz, float(fmax_hz)))
    return int(round(hz / float(fmax_hz) * (nF - 1)))

def band_hz_to_bins(band_hz: tuple, nF: int, fmax_hz: float) -> tuple:
    """Convert (f_lo_hz, f_hi_hz) -> (f0_bin, f1_bin) inclusive, clipped."""
    f_lo, f_hi = float(band_hz[0]), float(band_hz[1])
    if f_lo > f_hi:
        f_lo, f_hi = f_hi, f_lo
    b0 = hz_to_bin(f_lo, nF, fmax_hz)
    b1 = hz_to_bin(f_hi, nF, fmax_hz)
    if b0 > b1:
        b0, b1 = b1, b0
    return (b0, b1)

# -------------------------
# RUN TAGGING
# -------------------------
CLUSTER_TAG = _safe_tag(CLUSTER_COL) if CLUSTER_COL else "auto_cluster"
RUN_ID_RECON = f"{RUN_ID_230}__{CLUSTER_TAG}"

# -------------------------
# Determine ERSP shape (nF, nT) from 230 run
# -------------------------
P = R.ensure_dirs(RUN_ID_230, RUN_ID_RECON)
run230_dir = Path(P["run230_dir"])

ersp_npy = run230_dir / "ersp_keep.npy"
ersp_npz = run230_dir / "ersp_keep.npz"

if ersp_npy.exists():
    ersp_shape = np.load(ersp_npy, mmap_mode="r").shape  # (n, nF, nT)
elif ersp_npz.exists():
    z = np.load(ersp_npz)
    # best effort: infer nF,nT from first key
    ersp_shape = z[z.files[0]].shape
    # if it is per-sample arrays, it might be (nF,nT); then n is len(files)
    if len(ersp_shape) == 2:
        ersp_shape = (len(z.files), ersp_shape[0], ersp_shape[1])
else:
    raise FileNotFoundError(f"Missing ERSP stack: {ersp_npy} or {ersp_npz}")

_, nF, nT = int(ersp_shape[0]), int(ersp_shape[1]), int(ersp_shape[2])
print(f"[QC] ERSP shape: nF={nF}, nT={nT}")

# Convert Hz bands -> bin bands (inclusive)
F_BANDS_BINS = [band_hz_to_bins(b, nF=nF, fmax_hz=FMAX_HZ) for b in F_BANDS_HZ]
print("[QC] F_BANDS_BINS:", F_BANDS_BINS)

# -------------------------
# IMPORTANT: run build_atlas_inputs ONCE if coords cache not present
# (render_surface_activity_from_ersp_window needs coords_out under 240 atlas_cache)
# -------------------------
coords_path = Path(P["coords_out"])
if not coords_path.exists():
    print("[INFO] coords_out missing; running build_atlas_inputs(...) once.")
    R.build_atlas_inputs(RUN_ID_230, RUN_ID_RECON, cluster_col=CLUSTER_COL)

# -------------------------
# Main loop: time windows × frequency bands × condition
# -------------------------
t_last = nT - 1

for cond in CONDITIONS:
    cond_tag = "all" if cond is None else str(cond)

    for t0 in range(0, nT, T_INC):
        t1 = min(t0 + T_INC - 1, t_last)

        for (b0, b1), (hz0, hz1) in zip(F_BANDS_BINS, F_BANDS_HZ):
            print(f"[RUN] cond={cond_tag}  t={t0:03d}-{t1:03d}  fbin={b0:03d}-{b1:03d}  fHz={hz0}-{hz1}")

            R.render_surface_activity_from_ersp_window(
                RUN_ID_230,                           # 230 run ID: where df_keep_with_clusters + ersp_keep are loaded from
                RUN_ID_RECON,                         # 240 recon run ID: where outputs/caches/PNGs are written

                f_bins=(b0, b1),                      # frequency window to aggregate (must match your function’s convention: bin indices or Hz)
                t_bins=(t0, t1),                      # time window to aggregate (usually bin indices in your ERSP grid)
                condition=cond,                       # restrict samples to this condition (None = use all)

                vmin=-6,                              # lower color limit for activity (more negative saturates to deepest blue)
                vmax=6,                               # upper color limit for activity (more positive saturates to deepest red)
                cmap_name="bwr",                      # colormap (bwr makes 0 map to white by design)

                k_nearest=4,                          # how many nearest contacts contribute to each surface vertex (larger = smoother/more mixing)
                sigma_mm=4.0,                         # spatial smoothing scale on the surface in mm (larger = blurrier, more cancellation toward 0)

                density_saturation=True,              # apply density-based modulation (low coverage darker/attenuated; high coverage less attenuated)
                density_gamma=0.6,                    # density contrast curve (<1 boosts low-density differences; >1 compresses them)
                s_min=0.15,                           # minimum density scaling factor (lower = sparse regions get darker/weaker)
                s_max=1.25,                           # maximum density scaling factor (higher = dense regions can appear stronger/brighter)

                exclude_contacts_dist_to_pial_mm_gt=12.0,  # drop contacts farther than this from pial (reduces deep contacts driving surface color)
            )



[QC] ERSP shape: nF=129, nT=300
[QC] F_BANDS_BINS: [(0, 1), (1, 3), (3, 4), (4, 10), (10, 18), (18, 33), (33, 64), (64, 128)]
[RUN] cond=picture  t=000-029  fbin=000-001  fHz=0-5
[RUN] cond=picture  t=000-029  fbin=001-003  fHz=5-10
[RUN] cond=picture  t=000-029  fbin=003-004  fHz=10-16
[RUN] cond=picture  t=000-029  fbin=004-010  fHz=16-40
[RUN] cond=picture  t=000-029  fbin=010-018  fHz=40-70
[RUN] cond=picture  t=000-029  fbin=018-033  fHz=70-130
[RUN] cond=picture  t=000-029  fbin=033-064  fHz=130-250
[RUN] cond=picture  t=000-029  fbin=064-128  fHz=250-500
[RUN] cond=picture  t=030-059  fbin=000-001  fHz=0-5
[RUN] cond=picture  t=030-059  fbin=001-003  fHz=5-10
[RUN] cond=picture  t=030-059  fbin=003-004  fHz=10-16
[RUN] cond=picture  t=030-059  fbin=004-010  fHz=16-40
[RUN] cond=picture  t=030-059  fbin=010-018  fHz=40-70
[RUN] cond=picture  t=030-059  fbin=018-033  fHz=70-130
[RUN] cond=picture  t=030-059  fbin=033-064  fHz=130-250
[RUN] cond=picture  t=030-059  fbin=064-128  fH

Exception ignored in: <function PolyData.__del__ at 0x0000024D94750AF0>
Traceback (most recent call last):
  File "C:\Users\artoni\.conda\envs\LORA_FLM_env1\lib\site-packages\pyvista\core\pointset.py", line 1743, in __del__
    self._glyph_geom = None
  File "C:\Users\artoni\.conda\envs\LORA_FLM_env1\lib\site-packages\pyvista\core\utilities\misc.py", line 311, in __setattr__
    def __setattr__(self, key: str, value: Any) -> None:
KeyboardInterrupt: 


[RUN] cond=reading  t=000-029  fbin=064-128  fHz=250-500



KeyboardInterrupt



In [10]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFont

import functions.lf_blob_recon as R


# -----------------------------
# Parsing helpers (adjust if needed)
# -----------------------------
VIEWS_DEFAULT = ("left", "right")#, "frontal", "posterior", "dorsal", "ventral")


def _find_view_token(s: str, views: Tuple[str, ...]) -> Optional[str]:
    sl = s.lower()
    for v in views:
        if re.search(rf"(^|[_\-]){re.escape(v)}($|[_\-])", sl):
            return v
    return None


def _extract_pair(s: str, key: str) -> Optional[Tuple[float, float]]:
    """
    Extract two numbers from patterns like:
      f10-40, f10_40, f=10-40, fHz10-40, fbin10-40
      t10-20, tbin10-20, t=10-20
    """
    pat = rf"{key}(?:bin|hz)?[=_]?\s*([0-9]+(?:\.[0-9]+)?)\s*[-_]\s*([0-9]+(?:\.[0-9]+)?)"
    m = re.search(pat, s.lower())
    if not m:
        return None
    return float(m.group(1)), float(m.group(2))


def _extract_condition(path: Path) -> str:
    # try directory names first (common layout: .../<cond>/...)
    for part in path.parts[::-1]:
        p = str(part).strip().lower()
        if p in {"picture", "audio", "reading"}:
            return p
        if p.startswith("cond_"):
            return p.replace("cond_", "")
    # try filename tokens
    s = path.stem.lower()
    for c in ("picture", "audio", "reading"):
        if re.search(rf"(^|[_\-]){c}($|[_\-])", s):
            return c
    return "all"


def _phase_from_time_percent(t0_pct: float, t1_pct: float, split_pct: float = 50.0) -> str:
    # Simple rule: entirely before split -> stim; entirely after -> resp; else mixed
    if t1_pct <= split_pct:
        return "stim"
    if t0_pct >= split_pct:
        return "resp"
    return "mixed"


# -----------------------------
# Text overlay
# -----------------------------
def _load_font(size: int = 28) -> ImageFont.FreeTypeFont:
    # Prefer DejaVuSans if available (usually present via matplotlib installs); fallback to default.
    for p in [
        Path(r"C:\Windows\Fonts\arial.ttf"),
        Path(r"C:\Windows\Fonts\calibri.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"),
    ]:
        if p.exists():
            return ImageFont.truetype(str(p), size=size)
    return ImageFont.load_default()


def _composite_on_black(img: Image.Image) -> Image.Image:
    # MP4 has no alpha; if PNG is RGBA, blend onto black
    if img.mode in ("RGBA", "LA"):
        bg = Image.new("RGBA", img.size, (0, 0, 0, 255))
        bg.alpha_composite(img.convert("RGBA"))
        return bg.convert("RGB")
    return img.convert("RGB")


def _annotate_frame(
    img_path: Path,
    top_left: str,
    bottom_left: str,
    *,
    font: ImageFont.ImageFont,
    margin: int = 18,
) -> np.ndarray:
    img = Image.open(img_path)
    img = _composite_on_black(img)

    draw = ImageDraw.Draw(img)

    # High-contrast text: white with black stroke
    draw.text(
        (margin, margin),
        top_left,
        font=font,
        fill=(255, 255, 255),
        stroke_width=3,
        stroke_fill=(0, 0, 0),
    )

    # bottom-left
    w, h = img.size
    # estimate text height
    bbox = draw.textbbox((0, 0), bottom_left, font=font, stroke_width=3)
    th = (bbox[3] - bbox[1])
    draw.text(
        (margin, h - margin - th),
        bottom_left,
        font=font,
        fill=(255, 255, 255),
        stroke_width=3,
        stroke_fill=(0, 0, 0),
    )

    return np.asarray(img, dtype=np.uint8)


# -----------------------------
# Main: build MP4s from saved frames
# -----------------------------
def build_surface_activity_mp4s(
    RUN_ID_230: str,
    RUN_ID_RECON: str,
    *,
    out_subdir: str = "surface_activity",
    fps: int = 6,
    views: Tuple[str, ...] = VIEWS_DEFAULT,
    n_time_bins: Optional[int] = 300,   # used only if your t_bins are BIN INDICES
    fmax_hz: float = 500.0,             # used only if your f_bins are BIN INDICES
    overwrite: bool = False,
) -> Path:
    """
    Groups PNG frames into MP4s.

    Expected frame metadata is parsed from filenames/paths:
      - view token: one of `views`
      - f pair: e.g. f10-40 / fbin10-40 / fhz10-40
      - t pair: e.g. t0-30 / tbin0-30
      - condition from path tokens (picture/audio/reading) or filename tokens

    Writes:
      <run240>/reports/<out_subdir>_mp4/<condition>/<phase>/fXX-YY/<view>.mp4
    """
    P = R.ensure_dirs(RUN_ID_230, RUN_ID_RECON)
    frames_root = Path(P["reports_out"]) / out_subdir
    if not frames_root.exists():
        raise FileNotFoundError(f"Frames folder not found: {frames_root}")

    out_root = Path(P["reports_out"]) / f"{out_subdir}_mp4"
    out_root.mkdir(parents=True, exist_ok=True)

    # Optional: detect n_time_bins / n_freq_bins from ersp_keep.npy if present (for bin->% conversion)
    # We only need nT for percent labeling if your t values are bins.
    nT = n_time_bins
    try:
        run230_dir = Path(P["run230_dir"])
        f_ersp = run230_dir / "ersp_keep.npy"
        if f_ersp.exists():
            ersp = np.load(f_ersp, mmap_mode="r")
            if ersp.ndim == 3:
                nT = int(ersp.shape[2])
    except Exception:
        pass

    pngs = sorted(frames_root.rglob("*.png"))
    if not pngs:
        raise RuntimeError(f"No PNG frames found under: {frames_root}")

    # Group key: (condition, view, f0, f1, phase)
    groups: Dict[Tuple[str, str, float, float, str], List[Tuple[float, float, Path]]] = {}

    for p in pngs:
        s = str(p).lower()

        view = _find_view_token(p.name, views) or _find_view_token(str(p.parent), views)
        if view is None:
            continue

        fpair = _extract_pair(s, "f")
        tpair = _extract_pair(s, "t")
        if fpair is None or tpair is None:
            # If this happens, your filenames do not encode f/t ranges.
            # Best fix: include f/t in render filenames or write a manifest during rendering.
            continue

        f0, f1 = fpair
        t0, t1 = tpair
        cond = _extract_condition(p)

        # Convert t to percent if these look like bin indices
        # Heuristic: if nT known and t values are within [0, nT], treat as bins
        if nT and (0 <= t0 <= nT) and (0 <= t1 <= nT) and (abs(t1 - t0) >= 1):
            t0_pct = 100.0 * (t0 / max(1, (nT - 1)))
            t1_pct = 100.0 * (t1 / max(1, (nT - 1)))
        else:
            # already in percent
            t0_pct, t1_pct = float(t0), float(t1)

        phase = _phase_from_time_percent(t0_pct, t1_pct, split_pct=50.0)

        key = (cond, view, float(f0), float(f1), phase)
        groups.setdefault(key, []).append((t0_pct, t1_pct, p))

    if not groups:
        raise RuntimeError(
            "Found PNGs, but could not parse (f_bins, t_bins, view) from filenames/paths.\n"
            f"Frames root: {frames_root}\n"
            "Fix: ensure the render function encodes f/t/view in filenames, or write a manifest."
        )

    font = _load_font(size=30)

    # Write MP4s
    for (cond, view, f0, f1, phase), items in sorted(groups.items(), key=lambda kv: (kv[0][0], kv[0][4], kv[0][2], kv[0][1])):
        items = sorted(items, key=lambda x: x[0])  # sort by t0_pct
        band_tag = f"f{int(round(f0))}-{int(round(f1))}"
        out_dir = out_root / cond / phase / band_tag
        out_dir.mkdir(parents=True, exist_ok=True)

        out_mp4 = out_dir / f"{view}.mp4"
        if out_mp4.exists() and not overwrite:
            continue

        # Build frames with overlay
        with imageio.get_writer(str(out_mp4), fps=fps, codec="libx264", quality=8) as w:
            for t0_pct, t1_pct, p in items:
                top_left = f"{phase}|{band_tag}"
                bottom_left = f"{t0_pct:0.1f}–{t1_pct:0.1f}%"
                frame = _annotate_frame(p, top_left=top_left, bottom_left=bottom_left, font=font)
                w.append_data(frame)

        print(f"[WROTE] {out_mp4}  (frames={len(items)})")

    return out_root


if __name__ == "__main__":
    RUN_ID_230 = RUN_ID_230
    RUN_ID_RECON = RUN_ID_RECON  # set to your recon run id

    build_surface_activity_mp4s(
        RUN_ID_230,
        RUN_ID_RECON,
        out_subdir=Path("surface_activity","picture"),  # must match where your PNG frames are saved
        fps=6,
        overwrite=False,
    )

RuntimeError: Found PNGs, but could not parse (f_bins, t_bins, view) from filenames/paths.
Frames root: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\240_blob_cluster_recon\20260111_191040__cluster_kmeans_101_k23_q0p9\reports\surface_activity\picture
Fix: ensure the render function encodes f/t/view in filenames, or write a manifest.